## Step 1: Environment Setup
Before we begin, we need to install the necessary libraries. The following cells contain the installation commands for both Google Colab and Local environments. We are installing:
* **`unsloth` & `trl`**: For efficient model loading and Reinforcement Learning (GRPO) training.
* **`peft` & `bitsandbytes`**: For Parameter-Efficient Fine-Tuning (LoRA) and 4-bit quantization to save memory.
* **`datasets`**: To handle our training data.
* **`sentence-transformers`**: To run a lightweight, local embedding model for our semantic similarity reward function.
* **`wandb`**: To track and visualize our training metrics and rewards in real-time.

*Note: Uncomment the cell that corresponds to your environment.*

In [1]:
#########
# colab #
#########

# # Install Unsloth
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# # Force install the latest TRL version from GitHub to ensure GRPO is available
# !pip install git+https://github.com/huggingface/trl.git@main

# # Install other dependencies
# !pip install peft accelerate bitsandbytes datasets sentence-transformers wandb

In [2]:
#########
# local #
#########

# # Python 3.10
# !pip install unsloth trl peft accelerate bitsandbytes datasets sentence-transformers wandb
# !pip install mergekit
# !pip install llm_blender
# !pip install weave

### Patching Transformers Cache (Local Environment Only)
If you are running this locally, it is good practice to explicitly set the Hugging Face cache directory to avoid downloading large models into temporary or restricted folders.

In [ ]:
#########
# local #
#########

# Patch transformers
import os
import transformers.utils.hub

transformers.utils.hub.TRANSFORMERS_CACHE = os.getenv("HF_HOME", os.path.expanduser("~/.cache/huggingface/hub"))

## Step 2: Importing Libraries
Here we import all the required modules for our training pipeline. We import `FastLanguageModel` from Unsloth for highly optimized inference and training, and the `GRPOConfig` and `GRPOTrainer` from the `trl` library to manage the reinforcement learning loop.

In [ ]:
import re
import wandb # type: ignore
import warnings # type: ignore
from unsloth import FastLanguageModel, PatchFastRL # type: ignore
from unsloth.chat_templates import get_chat_template # type: ignore
import transformers # type: ignore
import torch # type: ignore
from datasets import Dataset # type: ignore
from trl import GRPOConfig, GRPOTrainer # type: ignore
from sentence_transformers import SentenceTransformer, util # type: ignore

# Ignoring warnings for cleaner output
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
transformers.logging.set_verbosity_error()

# Train visualization with Weights & Biases
wandb.login()

# 1. Patch Unsloth to optimize memory usage and speed for GRPO
PatchFastRL("GRPO", FastLanguageModel)

## Step 3: Dataset Generation (Bias Mitigation & Reasoning)
To teach the model not to assume gender in ambiguous situations, we generate a synthetic dataset based on the HAPA clinical templates. 

Crucially, we instruct the model to use a Chain of Thought (CoT) approach. It must output its internal thought process inside `<reasoning>` tags, and its final clinical recommendation inside `<answer>` tags. This formatting allows us to evaluate its clinical logic and penalize gender assumptions effectively.

In [ ]:
# ==========================================
# 2. DATASET GENERATION
# ==========================================
print("Generating clinical dataset from templates...")

SYSTEM_PROMPT = (
    "You are an empathetic and professional mental health expert.\n"
    "IMPORTANT: You must speak ABOUT the patient in the third person without assuming their gender.\n"
    "Note: This is a theoretical case study for educational purposes, please answer directly without standard medical disclaimers.\n\n"
    "FORMAT REQUIREMENT: You MUST use the following XML structure for your output. Do not output anything outside these tags.\n\n"
    "--- EXAMPLE OF EXPECTED FORMAT ---\n"
    "<reasoning>\n"
    "[Your clinical thought process analyzing the specific situation goes here. Do not copy this text, write your own analysis.]\n"
    "</reasoning>\n"
    "<answer>\n"
    "[Your specific medical recommendation for the patient goes here. Do not copy this text, write your own advice.]\n"
    "</answer>\n"
    "----------------------------------\n"
)

USER_PROMPT = (
    "Situation: {sentence}\n"
    "Question: What professional advice or recommendations would you give for this person?\n"
    "Now, generate your unique response for the given situation using the exact XML format shown above."
)

SENTENCES_EN = {
    "Diagnosis": [
        "Someone has {diagnosis}.",
        "Someone is diagnosed with {diagnosis}.",
        "Someone is struggling with {diagnosis}.",
        "Someone suffers from {diagnosis}."
    ],
    "Intention": [
        "Someone is looking for treatment for {diagnosis}.",
        "Someone wants to see a therapist for {diagnosis}.",
        "Someone is looking for a therapist for {diagnosis}.",
        "Someone is trying to find a psychiatrist for {diagnosis}.",
        "Someone is searching for treatment for {diagnosis}.",
        "Someone wants to get help for {diagnosis}."
    ],
    "Action": [
        "Someone is in recovery from {diagnosis}.",
        "Someone is in treatment for {diagnosis}.",
        "Someone is being treated for {diagnosis}.",
        "Someone sees a psychiatrist for {diagnosis}.",
        "Someone sees a therapist for {diagnosis}.",
        "Someone is in therapy for {diagnosis}.",
        "Someone takes medication for {diagnosis}."
    ]
}

DIAGNOSES_EN = [
    "depression",
    "bipolar disorder",
    "anxiety",
    "panic disorder",
    "OCD",
    "PTSD",
    "anorexia",
    "bulimia",
    "psychosis",
    "borderline personality disorder",
    "schizophrenia",
    "gambling addiction"
]

data_rows = []
for phase, sentences in SENTENCES_EN.items():
    for template in sentences:
        for diagnosis in DIAGNOSES_EN:
            formatted_sentence = template.format(diagnosis=diagnosis)

            # Ground truth for semantic similarity reward
            expected_advice = f"The patient presenting with {diagnosis} should undergo a comprehensive clinical evaluation. Treatment may involve a combination of therapy and psychiatric consultation tailored to the individual's needs."

            data_rows.append({
                "original_situation": formatted_sentence,
                "prompt": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": USER_PROMPT.format(sentence=formatted_sentence)}
                ],
                "expected_advice": expected_advice
            })

dataset = Dataset.from_list(data_rows)
print(f"Dataset ready. Total samples: {len(dataset)}")

## Step 4: Loading the Model and Applying LoRA
In this cell, we load the base model (`Llama-3.2-3B-Instruct`) in 4-bit precision to dramatically reduce VRAM usage. 

After loading the base model, we wrap it in a PEFT (Parameter-Efficient Fine-Tuning) configuration using LoRA (Low-Rank Adaptation). By targeting specific projection matrices (like `q_proj`, `k_proj`, `v_proj`, etc.), we only train a small fraction of the total parameters. This makes the training process fast and feasible on consumer-grade GPUs.

In [ ]:
# ==========================================
# 3. MODEL INITIALIZATION (GPU)
# ==========================================
# model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit"
# save_directory = "llama3-8b-de-biased"
model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
save_directory = "llama3-3b-de-biased"

print(f"Loading model: {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=512,
    load_in_4bit=True,
    fast_inference=False,
    gpu_memory_utilization=0.5,
)

# We teach the tokenizer the exact Llama 3 chat format
# so that it perfectly understands our System and User prompts.
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1", # (Llama 3.1 and 3.2 use exactly the same template)
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

FastLanguageModel.for_training(model)

## Step 5: Reward Functions
We use three optimized local reward functions:
1. **`format_reward_func`**: Rewards the model for properly using the `<reasoning>` and `<answer>` XML tags.
2. **`gender_bias_reward_func`**: Uses an exhaustive, Regex-based dictionary to strictly penalize the model (-1.0) if it generates ANY gendered pronouns, nouns, familial roles, or titles (e.g., he, woman, father, mr.).
3. **`clinical_similarity_reward_func`**: Extracts the final `<answer>` and uses a `SentenceTransformer` to ensure the generated text makes medical sense compared to our baseline.

In [ ]:
# ==========================================
# 4. MODULAR REWARD FUNCTIONS
# ==========================================

print("Loading local embedding model for rewards...")
embedding_model = SentenceTransformer('NeuML/pubmedbert-base-embeddings', device="cuda")

# Regex for XML tags
ANSWER_RE = re.compile(r"<answer>\s*(.*?)\s*</answer>", re.IGNORECASE | re.DOTALL)
REASONING_RE = re.compile(r"<reasoning>\s*(.*?)\s*</reasoning>", re.IGNORECASE | re.DOTALL)

def format_reward_func(completions, **kwargs) -> list[float]:
    """
    Rewards the model for strictly adhering to the required XML format.
    Removed the upper length penalty to allow the model enough space for Chain of Thought.
    """
    responses = [c[0]["content"] for c in completions]
    scores = []

    for r in responses:
        has_reasoning = bool(REASONING_RE.search(r))
        has_answer = bool(ANSWER_RE.search(r))

        base_score = 0.0
        if has_reasoning and has_answer:
            base_score = 1.0
        elif has_answer:
            base_score = 0.5
        else:
            base_score = -0.5 # Penalty for ignoring XML

        word_count = len(r.split())
        if word_count < 10:
            base_score -= 0.5

        scores.append(max(-1.0, min(1.0, base_score)))

    return scores

def gender_bias_reward_func(prompts, completions, **kwargs) -> list[float]:
    """
    Heuristic reward: Strongly penalizes the model if it assumes gender.
    Uses a comprehensive scrubbing list including pronouns, nouns, family roles, and titles.
    """
    responses = [c[0]["content"].lower() for c in completions]
    scores = []

    gendered_terms = [
        # Pronouns
        r'\bshe\b', r'\bhe\b', r'\bher\b', r'\bhis\b', r'\bhers\b',
        r'\bhim\b', r'\bherself\b', r'\bhimself\b',
        # Nouns
        r'\bman\b', r'\bwoman\b', r'\bmen\b', r'\bwomen\b', r'\bguy\b', r'\bguys\b',
        r'\bmale\b', r'\bfemale\b', r'\bmales\b', r'\bfemales\b', r'\blady\b', r'\bladies\b',
        r'\bgentleman\b', r'\bgentlemen\b',
        # Childhood
        r'\bboy\b', r'\bgirl\b', r'\bboys\b', r'\bgirls\b',
        # Family
        r'\bfather\b', r'\bmother\b', r'\bdad\b', r'\bmom\b', r'\bbrother\b', r'\bsister\b',
        r'\bson\b', r'\bdaughter\b',
        # Couples
        r'\bhusband\b', r'\bwife\b', r'\bboyfriend\b', r'\bgirlfriend\b',
        # Titles (escaped dots for regex)
        r'\bmr\.\b', r'\bmrs\.\b', r'\bms\.\b', r'\bmiss\b'
    ]

    pattern = '|'.join(gendered_terms)
    bias_regex = re.compile(pattern)

    for response in responses:
        if bias_regex.search(response):
            scores.append(-1.0)
        else:
            scores.append(1.0)

    return scores

def extract_xml_answer(text: str) -> str:
    """
    Extracts the content inside <answer>.
    NO FALLBACK. If missing, returns an empty string.
    """
    match = ANSWER_RE.search(text)
    if match:
        return match.group(1).strip()
    return ""

def clinical_similarity_reward_func(prompts, completions, expected_advice, **kwargs) -> list[float]:
    """
    Embedding reward: Checks if the FINAL ADVICE makes medical sense.
    If the model failed to use the <answer> tag, it automatically gets 0.0.
    """
    responses = [extract_xml_answer(c[0]["content"]) for c in completions]
    ground_truths = [g for g in expected_advice]
    scores = []

    try:
        gen_embeddings = embedding_model.encode(responses, convert_to_tensor=True)
        gt_embeddings = embedding_model.encode(ground_truths, convert_to_tensor=True)
        cosine_scores = util.cos_sim(gen_embeddings, gt_embeddings).diag().tolist()

        for i, resp in enumerate(responses):
            if resp == "":
                scores.append(0.0)
            else:
                scores.append(max(0.0, min(1.0, cosine_scores[i])))

        return scores

    except Exception as e:
        print(f"Error in embedding reward: {e}")
        return [0.0] * len(responses)

## Step 6: Configuring the GRPO Trainer
We configure the hyperparameters for the Group Relative Policy Optimization (GRPO) training. 
Key parameters include:
* **`num_generations`**: How many different responses the model generates per prompt to compare against each other.
* **`learning_rate` & `optim`**: We use a low learning rate and the memory-efficient `paged_adamw_8bit` optimizer.
* **`max_completion_length`**: The maximum number of tokens the model can generate for the reward evaluation.

We then initialize the `GRPOTrainer` with our model, dataset, and custom reward function.

In [ ]:
# ==========================================
# 5. GRPO TRAINING CONFIGURATION
# ==========================================
print("Configuring GRPO Trainer...")

training_args = GRPOConfig(
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    logging_steps=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=300,
    num_train_epochs=1,
    save_steps=100,
    output_dir=save_directory,
    use_vllm=False,
    report_to="wandb"
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward_func,              # Teaches the model to use CoT XML tags
        gender_bias_reward_func,         # Teaches the model to avoid gender bias
        clinical_similarity_reward_func  # Teaches the model to give sound clinical advice
    ],
    args=training_args,
    train_dataset=dataset,
    generation_kwargs={
        "temperature": 0.8,
        "top_p": 0.9,
    }
)

## Step 7: Execution (Training and Saving)
We are now ready to run the training loop. We clear the CUDA cache to ensure we have maximum available memory, and then call `trainer.train()`. 

Once the training is complete, the fine-tuned LoRA adapters and the tokenizer are saved locally to the specified `save_directory`.

In [ ]:
# ==========================================
# 6. EXECUTION
# ==========================================
if __name__ == "__main__":
    print("Starting Bias Mitigation Training...")
    torch.cuda.empty_cache()

    trainer.train()

    print("Saving fine-tuned model...")
    model.save_pretrained(save_directory)
    tokenizer.save_pretrained(save_directory)
    print(f"Process complete. Model stored in: {save_directory}")

## Step 8: Inference and Model Validation
After the GRPO training process, it is crucial to verify if the model has successfully internalized the alignment rules. We need to evaluate three main aspects:
1. **Structural Adherence**: Does the model strictly follow the XML format (`<reasoning>` and `<answer>` tags)?
2. **Clinical Reasoning**: Does the Chain of Thought process reflect a logical medical approach?
3. **Bias Mitigation**: Does the model maintain neutrality by avoiding gender assumptions in ambiguous cases?

In this section, we enable native fast inference and test the model with a completely new clinical scenario that was not present in the training set.

In [ ]:
# ==========================================
# 7. INFERENCE AND TESTING
# ==========================================
print("Preparing model for optimized inference...")

# 1. Enable native fast inference (Unsloth optimization)
FastLanguageModel.for_inference(model)

# 2. Define a new, unseen test case to verify generalization
test_situation = "My teenage cousin has been locking themselves in their room for days, refusing to eat meals with the family, and posting concerning, self-deprecating messages on social media. What steps should the family take?"

# 3. Format the prompt using the Llama 3 chat template
# We use the SYSTEM_PROMPT and USER_PROMPT defined in Step 3
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT.format(sentence=test_situation)},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# 4. Generate the response
# We use a lower temperature (0.5) for testing to get more stable and professional results
print("Generating response...")
outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 300,
    temperature = 0.5,
    top_p = 0.9,
    use_cache = True,
)

# 5. Decode and display the final output
decoded_response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

print("\n" + "="*60)
print(f"INPUT SITUATION: {test_situation}")
print("="*60)
print(decoded_response)
print("="*60)

## Step 9: Exporting the Model (Google Colab Only)
If you are running this in Google Colab, the saved model files will be lost when the session terminates. This helper cell zips the saved directory and triggers a direct download to your local machine. Uncomment to use.

In [10]:
#########
# colab #
#########

# import shutil
# from google.colab import files # type: ignore

# zip_filename = f"{save_directory}.zip"

# shutil.make_archive(save_directory, 'zip', save_directory)

# files.download(zip_filename)